### Ken Vellian
#### CSC 555: Mining Big Data
#### Final Project
#### Due: Nov 19, 2024 | Professor: Ahmed Abid

#### The purpose of this notebook is to use PySpark to explore and preprocess the data so we can stage it in S3 for Athena to call SQL queries against the data.
#### Afterwards, Random Forest Classifier will be used to predict fraudelent credit card transactions.

In [ ]:
sc.install_pypi_package("python-dateutil==2.8.2")
sc.install_pypi_package("seaborn")
sc.install_pypi_package("matplotlib")
sc.install_pypi_package("pandas")
sc.install_pypi_package("numpy")
sc.install_pypi_package("sklearn")
sc.install_pypi_package("boto3")
sc.install_pypi_package("time")

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
7,application_1732026294310_0008,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.8.1
    Not uninstalling python-dateutil at /usr/lib/python3.9/site-packages, outside environment /mnt/yarn/usercache/livy/appcache/application_1732026294310_0008/container_1732026294310_0008_01_000001/tmp/spark-0545b5ff-add5-436f-bf3d-fb94e936229c
    Can't uninstall 'python-dateutil'. No files were found to uninstall.





  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
  Preparing metadata (setup.py): started
  Preparing metadata (setu

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
import boto3
import time
from sklearn.metrics import roc_curve, auc

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, when
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.ml.tuning import TrainValidationSplit

from pyspark.ml.classification import *
from pyspark.ml.evaluation import *
from pyspark.sql.functions import *
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder



VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("CreditCardFraudAnalysis") \
    .getOrCreate()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
# Path to the CSV file from the S3 bucket
file_path = 's3://finalprojectbucket-kenvellian/data/card_transdata.csv'

# Read the CSV file into a Spark DataFrame
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Display the first few rows to verify it loaded correctly
df.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+------------------------------+------------------------------+---------------+---------+---------------+------------+-----+
|distance_from_home|distance_from_last_transaction|ratio_to_median_purchase_price|repeat_retailer|used_chip|used_pin_number|online_order|fraud|
+------------------+------------------------------+------------------------------+---------------+---------+---------------+------------+-----+
| 57.87785658389723|            0.3111400080477545|            1.9459399775518593|            1.0|      1.0|            0.0|         0.0|  0.0|
|10.829942699255545|           0.17559150228166587|            1.2942188106198573|            1.0|      0.0|            0.0|         0.0|  0.0|
| 5.091079490616996|            0.8051525945853258|           0.42771456119427587|            1.0|      0.0|            0.0|         1.0|  0.0|
|2.2475643282963613|              5.60004354707232|           0.36266257805709584|            1.0|      1.0|            0.0|         1.0

### Exploratory Data Analysis

In [ ]:
# Show the schema (column names and data types)
df.printSchema()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- distance_from_home: double (nullable = true)
 |-- distance_from_last_transaction: double (nullable = true)
 |-- ratio_to_median_purchase_price: double (nullable = true)
 |-- repeat_retailer: double (nullable = true)
 |-- used_chip: double (nullable = true)
 |-- used_pin_number: double (nullable = true)
 |-- online_order: double (nullable = true)
 |-- fraud: double (nullable = true)

In [ ]:
# Get summary statistics for numerical columns
df.describe().show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+--------------------+------------------------------+------------------------------+------------------+------------------+------------------+-------------------+-------------------+
|summary|  distance_from_home|distance_from_last_transaction|ratio_to_median_purchase_price|   repeat_retailer|         used_chip|   used_pin_number|       online_order|              fraud|
+-------+--------------------+------------------------------+------------------------------+------------------+------------------+------------------+-------------------+-------------------+
|  count|             1000000|                       1000000|                       1000000|           1000000|           1000000|           1000000|            1000000|            1000000|
|   mean|  26.628792192571254|             5.036519397861605|            1.8241819784044937|          0.881536|          0.350399|          0.100608|           0.650552|           0.087403|
| stddev|   65.39078439904455|            25.84309

In [ ]:
# Get the column names
print(f"Columns: {df.columns}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Columns: ['distance_from_home', 'distance_from_last_transaction', 'ratio_to_median_purchase_price', 'repeat_retailer', 'used_chip', 'used_pin_number', 'online_order', 'fraud']

In [ ]:
# Get the number of rows in the DataFrame
print(f"Number of rows: {df.count()}")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Number of rows: 1000000

In [ ]:
# Check for missing values in each column
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+------------------------------+------------------------------+---------------+---------+---------------+------------+-----+
|distance_from_home|distance_from_last_transaction|ratio_to_median_purchase_price|repeat_retailer|used_chip|used_pin_number|online_order|fraud|
+------------------+------------------------------+------------------------------+---------------+---------+---------------+------------+-----+
|                 0|                             0|                             0|              0|        0|              0|           0|    0|
+------------------+------------------------------+------------------------------+---------------+---------+---------------+------------+-----+

In [ ]:
# Show column names and their data types
print("Column names")
df.dtypes

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Column names
[('distance_from_home', 'double'), ('distance_from_last_transaction', 'double'), ('ratio_to_median_purchase_price', 'double'), ('repeat_retailer', 'double'), ('used_chip', 'double'), ('used_pin_number', 'double'), ('online_order', 'double'), ('fraud', 'double')]

In [ ]:
# Get the numerical columns from the DataFrame
numerical_columns = [field.name for field in df.schema.fields if isinstance(field.dataType, (IntegerType, DoubleType))]

# Loop through the numerical columns and calculate the min and max values for each
for column in numerical_columns:
    min_value = df.select(F.min(column)).first()[0]
    max_value = df.select(F.max(column)).first()[0]
    print(f"Column: {column}, Min: {min_value}, Max: {max_value}")


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Column: distance_from_home, Min: 0.004874385066744207, Max: 10632.723672241103
Column: distance_from_last_transaction, Min: 0.00011828161524401959, Max: 11851.104564946141
Column: ratio_to_median_purchase_price, Min: 0.004399239615964007, Max: 267.8029421832708
Column: repeat_retailer, Min: 0.0, Max: 1.0
Column: used_chip, Min: 0.0, Max: 1.0
Column: used_pin_number, Min: 0.0, Max: 1.0
Column: online_order, Min: 0.0, Max: 1.0
Column: fraud, Min: 0.0, Max: 1.0

In [ ]:

# Group by 'fraud' and calculate the average for other columns
df.groupBy("fraud").agg(
    F.avg("distance_from_home").alias("avg_distance_from_home"),
    F.avg("distance_from_last_transaction").alias("avg_distance_from_last_transaction"),
    F.avg("ratio_to_median_purchase_price").alias("avg_purchase_price")
).show()


In [ ]:
# Group by 'repeat_retailer' and count transactions
df.groupBy("repeat_retailer").count().show()

### Creating new feature: 'distance_ratio'-- Ratio of distance_from_home to distance_from_last_transaction

In [ ]:
# Create a new feature 'distance_ratio' as the ratio of distance_from_home and distance_from_last_transaction
df_with_new_feature = df.withColumn('distance_ratio', col('distance_from_home') / col('distance_from_last_transaction'))

# Show the updated DataFrame
df_with_new_feature.show(5)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+------------------+------------------------------+------------------------------+---------------+---------+---------------+------------+-----+------------------+
|distance_from_home|distance_from_last_transaction|ratio_to_median_purchase_price|repeat_retailer|used_chip|used_pin_number|online_order|fraud|    distance_ratio|
+------------------+------------------------------+------------------------------+---------------+---------+---------------+------------+-----+------------------+
| 57.87785658389723|            0.3111400080477545|            1.9459399775518593|            1.0|      1.0|            0.0|         0.0|  0.0|186.01868961517158|
|10.829942699255545|           0.17559150228166587|            1.2942188106198573|            1.0|      0.0|            0.0|         0.0|  0.0| 61.67691806567759|
| 5.091079490616996|            0.8051525945853258|           0.42771456119427587|            1.0|      0.0|            0.0|         1.0|  0.0| 6.323123746796136|
|2.2475643282963613|  

## Writing Transformed Data to S3

In [ ]:
# Define the output path for the transformed data to be saved in S3
output_file_path = 's3://finalprojectbucket-kenvellian/transformed_data/card_transdata_transformed.csv'

df_with_new_feature = df_with_new_feature.dropDuplicates()
# Coalesce the DataFrame to a single partition (if you want a single file output)
df_with_new_feature.coalesce(1).write.option("header", "true").mode("overwrite").csv(output_file_path)


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

## Staging card_transdata_transformed.csv in Athena through a new Table

In [ ]:
import os
# Now that the data is written to S3, we will create the Athena table
AWS_REGION = 'us-east-1'
S3_BUCKET_NAME = 'finalprojectbucket-kenvellian'
S3_KEY = 'transformed_data/'
TABLE_NAME = 'card_transdata_transformed2'

# AWS credentials (Replace with your live credentials)
AWS_ACCESS_KEY = os.environ["AWS_ACCESS_KEY_ID"]

AWS_SECRET_ACCESS_KEY = os.environ["AWS_SECRET_ACCESS_KEY"]

AWS_SESSION_TOKEN = os.environ["AWS_SESSION_TOKEN"]

In [ ]:
# Initialize Athena client with the credentials
athena_client = boto3.client(
    'athena',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_REGION
)

# Function to run Athena query
def run_athena_query(query, database, output_location):
    response = athena_client.start_query_execution(
        QueryString=query,
        QueryExecutionContext={'Database': database},
        ResultConfiguration={'OutputLocation': output_location}
    )
    return response['QueryExecutionId']

# Function to check query status
def check_query_status(query_execution_id):
    while True:
        response = athena_client.get_query_execution(QueryExecutionId=query_execution_id)
        status = response['QueryExecution']['Status']['State']
        
        # Check if the query has finished successfully
        if status == 'SUCCEEDED':
            print("Query succeeded!")
            break
        elif status in ['FAILED', 'CANCELLED']:
            print(f"Query {status.lower()}.")
            raise Exception(f"Query failed or was cancelled: {response}")
        
        # Wait for a short time before checking the status again
        time.sleep(2)
        
# Function to get query results and convert them to a pandas DataFrame
def get_query_results_as_dataframe(query_execution_id):
    results_paginator = athena_client.get_paginator('get_query_results')
    results_iter = results_paginator.paginate(QueryExecutionId=query_execution_id)

    columns = []
    rows = []

    for results_page in results_iter:
        if not columns:
            columns = [col['Label'] for col in results_page['ResultSet']['ResultSetMetadata']['ColumnInfo']]
        
        for row in results_page['ResultSet']['Rows'][1:]:  # Skip header row
            rows.append([col.get('VarCharValue', None) for col in row['Data']])

    df = pd.DataFrame(rows, columns=columns)
    return df

# Define the SQL query to create the Athena table (assuming CSV format)
create_table_query = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS `{TABLE_NAME}` (
    distance_from_home DOUBLE,
    distance_from_last_transaction DOUBLE,
    ratio_to_median_purchase_price DOUBLE,
    repeat_retailer INT,
    used_chip INT,
    used_pin_number INT,
    online_order INT,
    fraud INT,
    distance_ratio DOUBLE
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY ','
LOCATION 's3://{S3_BUCKET_NAME}/{S3_KEY}'
"""

# Print the query to ensure it's correct
print(create_table_query.strip())

# Execute the query to create the Athena table
response = athena_client.start_query_execution(
    QueryString=create_table_query.strip(),
    QueryExecutionContext={'Database': 'default'},
    ResultConfiguration={'OutputLocation': f's3://{S3_BUCKET_NAME}/query_results/'}
)

# Wait for the query to finish
print("Athena table creation query submitted.")
query_execution_id = response['QueryExecutionId']

# Run the Athena query
query = f'SELECT fraud, COUNT(*) as fraud_count FROM "{TABLE_NAME}" GROUP BY fraud;'
# query = f'DROP TABLE IF EXISTS "{TABLE_NAME}";'
query_execution_id = run_athena_query(query, 'default', f's3://{S3_BUCKET_NAME}/query_results/')

# Check the query status
check_query_status(query_execution_id)

# Fetch the query results into a pandas DataFrame
df = get_query_results_as_dataframe(query_execution_id)

# Display the results
df

# Class imbalance between fradulent vs non fradulent transactions.

### Fraudulent Transactions (fraud = 1): There are 87,403 transactions.
### Non-Fraudulent Transactions (fraud = 0): There are 912,597 transactions.
### Key Insights:
### The dataset is highly imbalanced with respect to the fraud column., our target.
### Approximately 91% of transactions are non-fraudulent and only 9% being fraudulent.

### Random Forest Classifier before class balancing

In [ ]:
# Prepare features vector
assembler = VectorAssembler(inputCols=['distance_from_home', 'distance_from_last_transaction', 
                                       'ratio_to_median_purchase_price', 'repeat_retailer', 
                                       'used_chip', 'used_pin_number', 'online_order', 'distance_ratio'], 
                            outputCol='features')

df_assembled = assembler.transform(df_with_new_feature)

# Now no class weighting (no additional column for classWeights)
# Create the Random Forest classifier (without class weights)
rf = RandomForestClassifier(labelCol="fraud", featuresCol="features")

# Set up the evaluator for AUC
evaluator = BinaryClassificationEvaluator(labelCol="fraud", metricName="areaUnderROC")

# Set up CrossValidator with 5 folds
crossval = CrossValidator(estimator=rf, 
                          estimatorParamMaps=ParamGridBuilder().build(),  # Param grid is empty here but can be added
                          evaluator=evaluator,
                          numFolds=5)

# Perform cross-validation
cv_model = crossval.fit(df_assembled)

# Get the best model from cross-validation
best_model = cv_model.bestModel

# Make predictions on the entire dataset using the best model
predictions = best_model.transform(df_assembled)

# Evaluate the best model using AUC-ROC
roc_auc = evaluator.evaluate(predictions)
print(f"AUC from cross-validation: {roc_auc}")

# Extract the predicted and actual values
prediction_and_labels = predictions.select("prediction", "fraud").rdd

# Create a metrics object for confusion matrix and accuracy
metrics = MulticlassMetrics(prediction_and_labels)

# Confusion Matrix
confusion_matrix = metrics.confusionMatrix().toArray()
print("Confusion Matrix:")
print(confusion_matrix)

# Accuracy
accuracy = metrics.accuracy
print(f"Accuracy: {accuracy}")

# Precision (for fraud = 1)
precision_fraud = metrics.precision(1.0)  # fraud=1
print(f"Precision for fraud = 1: {precision_fraud}")

# Precision (for non-fraud = 0)
precision_non_fraud = metrics.precision(0.0)  # non-fraud=0
print(f"Precision for non-fraud = 0: {precision_non_fraud}")

# Recall (for fraud = 1)
recall_fraud = metrics.recall(1.0)  # fraud=1
print(f"Recall for fraud = 1: {recall_fraud}")

# Recall (for non-fraud = 0)
recall_non_fraud = metrics.recall(0.0)  # non-fraud=0
print(f"Recall for non-fraud = 0: {recall_non_fraud}")

# F-Measure (for fraud = 1)
f1_fraud = metrics.fMeasure(1.0)  # fraud=1
print(f"F-Measure for fraud = 1: {f1_fraud}")

# F-Measure (for non-fraud = 0)
f1_non_fraud = metrics.fMeasure(0.0)  # non-fraud=0
print(f"F-Measure for non-fraud = 0: {f1_non_fraud}")


VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-20:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


AUC from cross-validation: 0.9897204565581632
Confusion Matrix:
[[912597.      0.]
 [ 12551.  74852.]]
Accuracy: 0.987449
Precision for fraud = 1: 1.0
Precision for non-fraud = 0: 0.9864335219878333
Recall for fraud = 1: 0.8564008100408452
Recall for non-fraud = 0: 1.0
F-Measure for fraud = 1: 0.922646451573141
F-Measure for non-fraud = 0: 0.993170434418268
/mnt/yarn/usercache/livy/appcache/application_1732026294310_0008/container_1732026294310_0008_01_000001/pyspark.zip/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.

### Random Forest Classifier after class balancing

In [ ]:
# Prepare features vector
assembler = VectorAssembler(inputCols=['distance_from_home', 'distance_from_last_transaction', 
                                       'ratio_to_median_purchase_price', 'repeat_retailer', 
                                       'used_chip', 'used_pin_number', 'online_order', 'distance_ratio'], 
                            outputCol='features')

df_assembled = assembler.transform(df_with_new_feature)

# Calculate class weights for imbalanced data
fraud_count = df_assembled.filter(df_assembled.fraud == 1).count()
non_fraud_count = df_assembled.filter(df_assembled.fraud == 0).count()
fraud_weight = non_fraud_count / fraud_count
non_fraud_weight = 1

# Add class weights to the DataFrame
df_with_weights = df_assembled.withColumn("classWeights", 
                                           when(df_assembled.fraud == 1, fraud_weight)
                                           .otherwise(non_fraud_weight))

# Create the Random Forest classifier with class weights
rf = RandomForestClassifier(labelCol="fraud", featuresCol="features", weightCol="classWeights")

# Set up the evaluator for AUC
evaluator = BinaryClassificationEvaluator(labelCol="fraud", metricName="areaUnderROC")

# Set up CrossValidator with 5 folds
crossval = CrossValidator(estimator=rf, 
                          estimatorParamMaps=ParamGridBuilder().build(),  # Param grid is empty here but can be added
                          evaluator=evaluator,
                          numFolds=5)

# Perform cross-validation
cv_model = crossval.fit(df_with_weights)

# Get the best model from cross-validation
best_model = cv_model.bestModel

# Make predictions on the entire dataset using the best model
predictions = best_model.transform(df_with_weights)

# Evaluate the best model using AUC-ROC
roc_auc = evaluator.evaluate(predictions)
print(f"AUC from cross-validation: {roc_auc}")

# Extract the predicted and actual values
prediction_and_labels = predictions.select("prediction", "fraud").rdd


# Create a metrics object for confusion matrix and accuracy
metrics = MulticlassMetrics(prediction_and_labels)

# Confusion Matrix
confusion_matrix = metrics.confusionMatrix().toArray()
print("Confusion Matrix:")
print(confusion_matrix)

# Accuracy
accuracy = metrics.accuracy
print(f"Accuracy: {accuracy}")

# Precision (for fraud = 1)
precision_fraud = metrics.precision(1.0)  # fraud=1
print(f"Precision for fraud = 1: {precision_fraud}")

# Precision (for non-fraud = 0)
precision_non_fraud = metrics.precision(0.0)  # non-fraud=0
print(f"Precision for non-fraud = 0: {precision_non_fraud}")

# Recall (for fraud = 1)
recall_fraud = metrics.recall(1.0)  # fraud=1
print(f"Recall for fraud = 1: {recall_fraud}")

# Recall (for non-fraud = 0)
recall_non_fraud = metrics.recall(0.0)  # non-fraud=0
print(f"Recall for non-fraud = 0: {recall_non_fraud}")

# F-Measure (for fraud = 1)
f1_fraud = metrics.fMeasure(1.0)  # fraud=1
print(f"F-Measure for fraud = 1: {f1_fraud}")

# F-Measure (for non-fraud = 0)
f1_non_fraud = metrics.fMeasure(0.0)  # non-fraud=0
print(f"F-Measure for non-fraud = 0: {f1_non_fraud}")



VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Exception in thread cell_monitor-21:
Traceback (most recent call last):
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 980, in _bootstrap_inner
    self.run()
  File "/mnt/notebook-env/lib/python3.9/threading.py", line 917, in run
    self._target(*self._args, **self._kwargs)
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in cell_monitor
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
  File "/mnt/notebook-env/lib/python3.9/site-packages/awseditorssparkmonitoringwidget/cellmonitor.py", line 154, in <listcomp>
    job_group_filtered_jobs = [job for job in jobs_data if job['jobGroup'] == str(statement_id)]
KeyError: 'jobGroup'


AUC from cross-validation: 0.9945464068683745
Confusion Matrix:
[[895468.  17129.]
 [  1521.  85882.]]
Accuracy: 0.98135
Precision for fraud = 1: 0.8337167875275456
Precision for non-fraud = 0: 0.9983043270318811
Recall for fraud = 1: 0.9825978513323341
Recall for non-fraud = 0: 0.9812304883754823
F-Measure for fraud = 1: 0.9020555211276481
F-Measure for non-fraud = 0: 0.9896937752613028